# 02c · Calibration: three-way verdict, out-of-distribution check, CNN + CLIP fusion

Turns raw scores into a verdict you can trust, **using validation data only** (no test set, no DFDC):

- **Temperature scaling:** one number T per model, fitted so that P(fake) = sigmoid(logit / T) is calibrated
  (a 0.9 means about 90% of such images are fake). v1 models were badly over-confident: their thresholds sat
  around 0.9–0.98.
- **Three-way verdict:** `t_fake` is set so that at most `MAX_FPR` of validation real faces are called **FAKE**;
  `t_real` so that at most `MAX_FNR` of validation fakes are called **REAL**; in between the verdict is
  **UNCERTAIN**. This is what removes most false alarms on real photos.
- **Out-of-distribution flag:** Mahalanobis distance of an image's features to the training features; beyond
  the 99th percentile of validation distances the verdict is flagged "low trust".
- **Fusion:** for every CNN, a logistic regression on [CNN logit / T, CLIP logit], with its own thresholds.

**Notebook settings (right sidebar)**
- **Accelerator:** None (CPU) · **Internet:** On
- **Inputs:** **02a** (and every copy of it) and **02b**

**Produces:** `/kaggle/working/outputs/calibration/` (`calibration.json` + `ood_<model>.npz`). Attach it to **03**.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, checkpoint_roots, find_clip_probe
setup(data=False)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOTS = checkpoint_roots()
PROBE = find_clip_probe()
print("checkpoint folders:", ROOTS)
print("CLIP probe:", PROBE)
assert ROOTS, "Add the output of 02a as an input"
if PROBE is None:
    print("WARNING: 02b not attached: no CLIP calibration and no fusion")

## Settings

In [ ]:
MAX_FPR = 0.05   # share of validation real faces allowed to be called FAKE
MAX_FNR = 0.10   # share of validation fakes allowed to be called REAL
OOD_QUANTILE = 0.99

In [ ]:
args = " ".join(f'"{r}"' for r in ROOTS)
clip = f'--clip "{PROBE.parent}"' if PROBE else ""
!python -m src.classifier.calibrate --roots {args} {clip} --out /kaggle/working/outputs/calibration --max-fpr {MAX_FPR} --max-fnr {MAX_FNR} --ood-quantile {OOD_QUANTILE}

## Calibration per model
`ece_raw → ece_calibrated`: calibration error before/after temperature scaling (lower is better).
`val_rates`: on validation, the share UNCERTAIN, reals called FAKE (`fpr`), fakes called REAL (`fnr`) and accuracy on
the images that got a REAL/FAKE verdict.

In [ ]:
cal = json.loads(Path("/kaggle/working/outputs/calibration/calibration.json").read_text())
rows = []
for name, m in cal["models"].items():
    rows.append({"model": name, "T": m["temperature"], "t_real": m["t_real"], "t_fake": m["t_fake"],
                 "ece_raw": m["ece_raw"], "ece_calibrated": m["ece_calibrated"], **m["val_rates"],
                 "ood_threshold": m["ood_threshold"]})
if cal["clip"]:
    c = cal["clip"]
    rows.append({"model": "CLIP probe", "T": c["temperature"], "t_real": c["t_real"], "t_fake": c["t_fake"],
                 "ece_raw": c["ece_raw"], "ece_calibrated": c["ece_calibrated"], **c["val_rates"]})
display(pd.DataFrame(rows).set_index("model").round(4))
if cal["fused"]:
    display(Markdown("**CNN + CLIP fusion** (weights of the calibrated CNN logit and the CLIP logit)"))
    display(pd.DataFrame({k: {"w_cnn": v["w_cnn"], "w_clip": v["w_clip"], "t_real": v["t_real"], "t_fake": v["t_fake"],
                              "ece": v["ece"], **v["val_rates"]} for k, v in cal["fused"].items()}).T.round(4))
display(Markdown("**Out-of-distribution rate on validation, per dataset** (should be ~1% for the training datasets)"))
display(pd.DataFrame({k: v["ood_rate_val"] for k, v in cal["models"].items()}).T.round(3))

## Reliability diagram (main model, validation)

In [ ]:
from src.classifier.calibrate import find_val_preds, sigmoid
from src.config import SYNTHETIC

folders = find_val_preds(ROOTS)
main = "efficientnet_b0" if "efficientnet_b0" in folders else next(iter(folders))
z = np.load(folders[main] / "val_preds.npz")
keep = ~np.isin(z["datasets"], SYNTHETIC)
y, logit = z["labels"][keep], z["logits"][keep]
T = cal["models"][main]["temperature"]
bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(4.8, 4.5))
for p, lab in ((sigmoid(logit), "raw"), (sigmoid(logit / T), f"temperature-scaled (T={T:.2f})")):
    idx = np.clip(np.digitize(p, bins[1:-1]), 0, 9)
    xs = [p[idx == b].mean() for b in range(10) if (idx == b).any()]
    ys = [y[idx == b].mean() for b in range(10) if (idx == b).any()]
    ax.plot(xs, ys, marker="o", label=lab)
ax.plot([0, 1], [0, 1], "--", color="gray"); ax.set_xlabel("predicted P(fake)"); ax.set_ylabel("observed share fake")
ax.set_title(f"{main}: reliability on validation"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()